# Load the libraries and dataset

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import GridSearchCV
import joblib

In [2]:
dataset_path = Path("../data/processed/dataset.csv")
df = pd.read_csv(dataset_path)
df.head()

,Hours_Studied,Attendance,Parental_Involvement,Access_to_Resources,Extracurricular_Activities,Sleep_Hours,Previous_Scores,Motivation_Level,Internet_Access,Tutoring_Sessions,Family_Income,Teacher_Quality,School_Type,Peer_Influence,Physical_Activity,Learning_Disabilities,Parental_Education_Level,Distance_from_Home,Gender,Exam_Score
0,23,84,Low,High,False,7,73,Low,True,0,Low,Medium,Public,Positive,3,False,High School,Near,Male,67
1,19,64,Low,Medium,False,8,59,Low,True,2,Medium,Medium,Public,Negative,4,False,College,Moderate,Female,61
2,24,98,Medium,Medium,True,7,91,Medium,True,2,Medium,Medium,Public,Neutral,4,False,Postgraduate,Near,Male,74
3,29,89,Low,Medium,True,8,98,Medium,True,1,Medium,Medium,Public,Negative,4,False,High School,Moderate,Male,71
4,19,92,Medium,Medium,True,6,65,Medium,True,3,Medium,High,Public,Neutral,4,False,College,Near,Female,70


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6607 entries, 0 to 6606
Data columns (total 20 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   Hours_Studied               6607 non-null   int64
 1   Attendance                  6607 non-null   int64
 2   Parental_Involvement        6607 non-null   str  
 3   Access_to_Resources         6607 non-null   str  
 4   Extracurricular_Activities  6607 non-null   bool 
 5   Sleep_Hours                 6607 non-null   int64
 6   Previous_Scores             6607 non-null   int64
 7   Motivation_Level            6607 non-null   str  
 8   Internet_Access             6607 non-null   bool 
 9   Tutoring_Sessions           6607 non-null   int64
 10  Family_Income               6607 non-null   str  
 11  Teacher_Quality             6607 non-null   str  
 12  School_Type                 6607 non-null   str  
 13  Peer_Influence              6607 non-null   str  
 14  Physical_Activity  

# Train Test Split

In [4]:
X = df.drop("Exam_Score", axis=1)
y = df["Exam_Score"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Encode and Scaling

In [5]:
teacher_pipeline = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OrdinalEncoder(categories=[["Low", "Medium", "High"]])),
    ]
)

parent_edu_pipeline = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "encoder",
            OrdinalEncoder(categories=[["High School", "College", "Postgraduate"]]),
        ),
    ]
)

distance_pipeline = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OrdinalEncoder(categories=[["Near", "Moderate", "Far"]])),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("teacher", teacher_pipeline, ["Teacher_Quality"]),
        ("parent", parent_edu_pipeline, ["Parental_Education_Level"]),
        ("distance", distance_pipeline, ["Distance_from_Home"]),
        (
            "ordinal_enc",
            OrdinalEncoder(
                categories=[
                    ["Low", "Medium", "High"],
                    ["Low", "Medium", "High"],
                    ["Low", "Medium", "High"],
                    ["Low", "Medium", "High"],
                    ["Negative", "Neutral", "Positive"],
                ]
            ),
            [
                "Parental_Involvement",
                "Access_to_Resources",
                "Motivation_Level",
                "Family_Income",
                "Peer_Influence",
            ],
        ),
        (
            "ohe",
            OneHotEncoder(sparse_output=False, drop="first"),
            [
                "Extracurricular_Activities",
                "Internet_Access",
                "School_Type",
                "Learning_Disabilities",
                "Gender",
            ],
        ),
    ],
    remainder="passthrough",
)

pipeline = Pipeline([("preprocessor", preprocessor), ("scaler", StandardScaler())])

In [6]:
X_test.shape

(1322, 19)

# Compare and Choose Model

In [7]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(),
    "Lasso": Lasso(),
    "Decision Tree": DecisionTreeRegressor(
        max_depth=5, min_samples_split=10, min_samples_leaf=5, random_state=42
    ),
    "KNN": KNeighborsRegressor(),
}

for name, model in models.items():
    pipe = Pipeline(
        [("preprocessor", preprocessor), ("scaler", StandardScaler()), ("model", model)]
    )

    pipe.fit(X_train, y_train)

    train_pred = pipe.predict(X_train)
    test_pred = pipe.predict(X_test)

    print(f"\n{name}")
    print(f"Train R² : {r2_score(y_train, train_pred):.4f}")
    print(f"Test R²  : {r2_score(y_test, test_pred):.4f}")
    print(f"MAE      : {mean_absolute_error(y_test, test_pred):.2f}")
    print(f"RMSE     : {np.sqrt(mean_squared_error(y_test, test_pred)):.2f}")


Linear Regression
Train R² : 0.7169
Test R²  : 0.7709
MAE      : 0.44
RMSE     : 1.80

Ridge
Train R² : 0.7169
Test R²  : 0.7709
MAE      : 0.44
RMSE     : 1.80

Lasso
Train R² : 0.3990
Test R²  : 0.4378
MAE      : 1.91
RMSE     : 2.82

Decision Tree
Train R² : 0.5426
Test R²  : 0.5461
MAE      : 1.59
RMSE     : 2.53

KNN
Train R² : 0.6502
Test R²  : 0.5231
MAE      : 1.60
RMSE     : 2.60


In [8]:
pipe = Pipeline(
    [("preprocessor", preprocessor), ("scaler", StandardScaler()), ("model", Ridge())]
)

param_grid = {"model__alpha": [0.001, 0.01, 0.1, 1, 3, 5, 8, 10, 15, 20, 50, 80, 100]}

grid = GridSearchCV(
    estimator=pipe, param_grid=param_grid, cv=5, scoring="r2", n_jobs=-1
)

grid.fit(X_train, y_train)

best_alpha = grid.best_params_['model__alpha']
print("Best alpha:", best_alpha)
print("Best CV Score:", grid.best_score_)
print("Test Score:", grid.score(X_test, y_test))

Best alpha: 10
Best CV Score: 0.7236576486597986
Test Score: 0.7709419631026844


In [9]:
final_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('scaler', StandardScaler()),
    ('model', Ridge(alpha=best_alpha))
])

final_pipeline.fit(X, y)   # Fit on the entire dataset

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('scaler', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](19,)","['Hours_Studied','Attendance','Parental_Involvement',..., 'Parental_Education_Level','Distance_from_Home','Gender']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,19
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('teacher', ...), ('parent', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit

In [10]:
model_path = Path("../models")
model_path.mkdir(parents=True, exist_ok=True)
joblib.dump(final_pipeline, model_path / "student_exam_score_model.pkl")

['../models/student_exam_score_model.pkl']